In [2]:
import sys
from pathlib import Path

# To import the variables in config.py
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f 
from config import OFF_CSV_PATH, OFF_CSV_URL
import urllib.request

In [4]:
spark = SparkSession.builder.appName(
        "lakehouse-off").master("local[*]").getOrCreate() 

#### Read Dataset from https url

In [5]:
if not OFF_CSV_PATH.exists():
    # Create the data/raw dir
    OFF_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Download the csv file only when it doesn't exist 
    urllib.request.urlretrieve(OFF_CSV_URL, OFF_CSV_PATH)

In [6]:
# Verify the file path and its size. 
print(OFF_CSV_PATH.exists())
print(OFF_CSV_PATH.stat().st_size)

True
1261161707


In [7]:
df = spark.read.csv(
    str(OFF_CSV_PATH),
    sep = "\t",
    header=True,
    multiLine=True,
    quote='"',
    escape='"',
    encoding="utf-8"
    )

In [8]:
df.show(10)

+------------+--------------------+--------------------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+--------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+--------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+----------------+-------------------+-------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+---------------+--------------------+----------------+----------------+----------+-------------+----------------

In [9]:
df.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- creator: string (nullable = true)
 |-- created_t: string (nullable = true)
 |-- created_datetime: string (nullable = true)
 |-- last_modified_t: string (nullable = true)
 |-- last_modified_datetime: string (nullable = true)
 |-- last_modified_by: string (nullable = true)
 |-- last_updated_t: string (nullable = true)
 |-- last_updated_datetime: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- abbreviated_product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- packaging: string (nullable = true)
 |-- packaging_tags: string (nullable = true)
 |-- packaging_en: string (nullable = true)
 |-- packaging_text: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- brands_tags: string (nullable = true)
 |-- brands_en: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: s

In [10]:
print(f"There are {len(df.columns)} columns in the whole data set.")
print(f"There are {df.count()} rows in the whold data set.")

There are 210 columns in the whole data set.
There are 4475958 rows in the whold data set.


In [11]:
numeric_cols = [col for col in df.columns if "100g" in col]
len(numeric_cols)
print(numeric_cols)

['energy-kj_100g', 'energy-kcal_100g', 'energy_100g', 'energy-from-fat_100g', 'fat_100g', 'saturated-fat_100g', 'butyric-acid_100g', 'caproic-acid_100g', 'caprylic-acid_100g', 'capric-acid_100g', 'lauric-acid_100g', 'myristic-acid_100g', 'palmitic-acid_100g', 'stearic-acid_100g', 'arachidic-acid_100g', 'behenic-acid_100g', 'lignoceric-acid_100g', 'cerotic-acid_100g', 'montanic-acid_100g', 'melissic-acid_100g', 'unsaturated-fat_100g', 'monounsaturated-fat_100g', 'omega-9-fat_100g', 'polyunsaturated-fat_100g', 'omega-3-fat_100g', 'omega-6-fat_100g', 'alpha-linolenic-acid_100g', 'eicosapentaenoic-acid_100g', 'docosahexaenoic-acid_100g', 'linoleic-acid_100g', 'arachidonic-acid_100g', 'gamma-linolenic-acid_100g', 'dihomo-gamma-linolenic-acid_100g', 'oleic-acid_100g', 'elaidic-acid_100g', 'gondoic-acid_100g', 'mead-acid_100g', 'erucic-acid_100g', 'nervonic-acid_100g', 'trans-fat_100g', 'cholesterol_100g', 'carbohydrates_100g', 'sugars_100g', 'added-sugars_100g', 'sucrose_100g', 'glucose_100g

#### Exploring data by creating a sample 

In [12]:
# create a sample DataFrame
df_sample = df.sample(fraction=0.01, seed=42)
total_sample = df_sample.count()

##### Check the Null values in column level 

In [13]:
# Before counting Null values, make sure to replace some string values like "unknown", "N/A" or "" values into NULL value
def replace_missing_values(df):
    string_cols = [c for c, t in df.dtypes if t == "string"]

    return df.select([
        f.when(
            f.trim(f.lower(f.col(c))).isin("unknown", "n/a", "na", ""),
            None
        ).otherwise(f.col(c)).alias(c)
        if c in string_cols else f.col(c)
        for c in df.columns
    ])
    

In [14]:
df_sample_1 = replace_missing_values(df_sample)

In [63]:
df_sample_1.show(5)

+-------------+--------------------+--------------------+----------+--------------------+---------------+----------------------+--------------------+--------------+---------------------+--------------------+------------------------+------------+--------+---------+--------------+------------+--------------+----------------+----------------+----------------+--------------------+--------------------+--------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+----------------+----------------+-------------+----------------+----------------+-------------------------+---------+------------+------+-----------+---------+---------------+----------------+-----------------+-----------+---------+--------------+------------+----------------+----------------+----------+-------------+-------------+-----------

In [15]:
# Check the percentage of Null values in each column and pivot the DataFrame 
null_rates = df_sample_1.select([
f.round(
    (f.count(f.when(f.col(c).isNull(), 1)) / total_sample), 2
).alias(c)
    for c in df_sample.columns
])

df_null = null_rates.select(
    f.explode(
        f.array(
            [f.struct(
                    f.lit(c).alias("col_name"),
                    f.col(c).alias("null_rate")
                ) for c in null_rates.columns]
        )
    ).alias("tmp")
).select(
    "tmp.col_name",
    "tmp.null_rate"
    ).orderBy(
        f.col("null_rate").desc())

In [16]:
df_null.show(210, truncate=False)

+--------------------------------+---------+
|col_name                        |null_rate|
+--------------------------------+---------+
|cities                          |1.0      |
|allergens_en                    |1.0      |
|no_nutrition_data               |1.0      |
|additives                       |1.0      |
|energy-from-fat_100g            |1.0      |
|butyric-acid_100g               |1.0      |
|caproic-acid_100g               |1.0      |
|caprylic-acid_100g              |1.0      |
|capric-acid_100g                |1.0      |
|lauric-acid_100g                |1.0      |
|myristic-acid_100g              |1.0      |
|palmitic-acid_100g              |1.0      |
|stearic-acid_100g               |1.0      |
|arachidic-acid_100g             |1.0      |
|behenic-acid_100g               |1.0      |
|lignoceric-acid_100g            |1.0      |
|cerotic-acid_100g               |1.0      |
|montanic-acid_100g              |1.0      |
|melissic-acid_100g              |1.0      |
|unsaturat

##### Keep useful columns 

After inspecting the null values existing in the sample, it's better to select the useful columns rather than to drop the uesless ones. 

In [64]:
def trim_columns_and_drop_product_code_null(df):

    cols_to_keep = [
        "code",
        "last_modified_t",
        "product_name",
        "brands",
        "categories_en",
        "labels_en",
        "countries_en",
        "nutriscore_grade",
        "ingredients_tags",
        "food_groups_en",
        "energy-kcal_100g",
        "fat_100g",
        "saturated-fat_100g",
        "carbohydrates_100g",
        "sugars_100g",
        "proteins_100g",
        "salt_100g"
    ]
    return df.select([c for c in cols_to_keep if c in df.columns]).filter(
            f.col("product_name").isNotNull()
            & f.col("code").isNotNull()
            & (f.col("product_name") != "")
            & (f.col("code") != ""))

In [65]:
df_clean = trim_columns_and_drop_product_code_null(df_sample_1)

In [66]:
df_clean.show(5)

+-------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+----------------+--------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|         code|last_modified_t|        product_name|              brands|       categories_en|           labels_en| countries_en|nutriscore_grade|ingredients_tags|food_groups_en|energy-kcal_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|
+-------------+---------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+----------------+--------------+----------------+--------+------------------+------------------+-----------+-------------+---------+
|     00000269|     1728035306|Bulletproof colla...|         Bulletproof|Collagen,Collagen...|                NULL|      Ireland|            NULL|            NULL|

#### Apply the transformation applied to the sample to the original dataset

In [ ]:
df_replaced = replace_missing_values(df)
df_cols_trimmed = trim_columns_and_drop_product_code_null(df_replaced)

In [47]:
df_cols_trimmed.show(5)

+------------+--------------------+--------------+--------------------+--------------------+-------------+----------------+--------------------+--------------------+----------------+-----------+--------+------------------+------------------+-----------+-------------+---------+
|        code|        product_name|        brands|       categories_en|           labels_en| countries_en|nutriscore_grade|    ingredients_tags|      food_groups_en|energy-kcal_100g|energy_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|
+------------+--------------------+--------------+--------------------+--------------------+-------------+----------------+--------------------+--------------------+----------------+-----------+--------+------------------+------------------+-----------+-------------+---------+
|000000000054|Limonade artisana...|          NULL|                NULL|                NULL|       France|            NULL|                NULL|                NULL| 

#### Check the Null values in row level 

In [ ]:
col_to_check = [c for c in df_cols_trimmed.columns if c not in ["product_name", "code"]]
nb_col = len(col_to_check)

df_rows_null = df_cols_trimmed.withColumn(
        "null_count",
        sum(f.col(c).isNull().cast("int") for c in col_to_check)
    ).withColumn(
        "null_perc",
        f.round(f.col("null_count") / nb_col, 2)
    ).orderBy(f.col("null_perc").desc())

In [49]:
df_rows_null.show(5)

+-------------+--------------------+------+-------------+---------+------------+----------------+----------------+--------------+----------------+-----------+--------+------------------+------------------+-----------+-------------+---------+----------+---------+
|         code|        product_name|brands|categories_en|labels_en|countries_en|nutriscore_grade|ingredients_tags|food_groups_en|energy-kcal_100g|energy_100g|fat_100g|saturated-fat_100g|carbohydrates_100g|sugars_100g|proteins_100g|salt_100g|null_count|null_perc|
+-------------+--------------------+------+-------------+---------+------------+----------------+----------------+--------------+----------------+-----------+--------+------------------+------------------+-----------+-------------+---------+----------+---------+
|0000110010371|        Dice vanilla|  NULL|         NULL|     NULL|        NULL|            NULL|            NULL|          NULL|            NULL|       NULL|    NULL|              NULL|              NULL|      

In [53]:
df_rows_clean = df_rows_null.filter(f.col("null_perc") < 1)

In [54]:
df_rows_clean.count()

4138370